# 📊 ACfX: Causal Feature Exploration on Health Datasets

Welcome to this Jupyter Notebook showcasing **ACfX (Automated Causal Feature Exploration)** applied to two widely used health datasets:

1. **Breast Cancer Dataset**  
   - Binary classification problem: *malignant* vs *benign* tumors.  
   - 30 numeric features describing tumor characteristics such as radius, texture, perimeter, and smoothness.  
   - Well-known dataset from `sklearn.datasets.load_breast_cancer`.

2. **Diabetes (Pima) Dataset**  
   - Binary classification problem: *diabetic* vs *non-diabetic* cases.  
   - 8 numeric clinical features, including glucose, blood pressure, BMI, age, and pregnancies.  
   - Fetched via `sklearn.datasets.fetch_openml(name="diabetes")`.

---

## 🔍 Purpose of this Notebook

The goal of this notebook is to:

- **Load and preprocess** the datasets.
- **Split** into training and testing sets with stratification for reproducibility.
- **Explore causal relationships** among features using ACfX.
- **Visualize causal graphs** to uncover dependencies and potential actionable insights.

By the end of this notebook, you will be able to:

- Identify **key causal relationships** in the data.  
- Understand **feature importance from a causal perspective**, not just correlation.  
- Generate **graphical visualizations** of the learned causal structures.  

---

### ⚙️ Configuration

Before diving in, all parameters for data loading, test/train splitting, and causal analysis are defined at the top of the notebook so you can **easily adjust the behavior** without modifying the core functions.



## Install packages
**Note** ACFX relies on numpy 1.26.0, which is not natively present in Colab. After installation **Session restart is needed** -- You will be prompt for that.

In [ ]:
!python --version

In [ ]:
!pip show acfx

In [ ]:
# !pip install -q "git+https://github.com/sbobek/acfx.git@develop#subdirectory=src"

In [ ]:
!pip install -q lingam openml

In [ ]:
from acfx import AcfxEBM
from interpret.glassbox import ExplainableBoostingClassifier

model = ExplainableBoostingClassifier()
explainer = AcfxEBM(model)

In [ ]:
# ============================================================
# CONFIGURATION PARAMETERS (USER-ADJUSTABLE)
# ============================================================

# Dataset to load: "breast_cancer" or "diabetes"
DATASET_NAME = "diabetes"

# Fraction of the dataset to use as the test split
TEST_SIZE = 0.2

# Random seed for reproducibility
RANDOM_STATE = 42

# Whether to stratify the train/test split by the target variable
STRATIFY_SPLIT = True


# ============================================================
# IMPORTS
# ============================================================

from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd


# ============================================================
# DATA LOADING AND SAMPLING FUNCTION
# ============================================================

def sample_data(
    dataset=DATASET_NAME,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=STRATIFY_SPLIT,
):
    """
    Load a dataset (Breast Cancer or Pima Diabetes) and split into train/test sets.

    Parameters
    ----------
    dataset : str
        "breast_cancer" or "diabetes"
    test_size : float
        Fraction of data to reserve for testing
    random_state : int
        Random seed for reproducibility
    stratify : bool
        Whether to stratify the split using the target labels

    Returns
    -------
    X_train, X_test : pandas.DataFrame
        Training and test features
    y_train, y_test : pandas.Series or numpy array
        Training and test target
    feature_names : list of str
        List of feature names
    target_names : list of str
        List of target class names
    """

    if dataset == "breast_cancer":
        data = load_breast_cancer(as_frame=True)
        X = data.data
        y = data.target

        feature_names = X.columns.tolist()
        target_names = ["malignant (0)", "benign (1)"]

    elif dataset == "diabetes":
        diabetes = fetch_openml(name="diabetes", version=1, as_frame=True)

        X = diabetes.data

        le = LabelEncoder()
        y = le.fit_transform(diabetes.target.astype(str))

        feature_names = X.columns.tolist()
        target_names = ["non-diabetic (0)", "diabetic (1)"]

    else:
        raise ValueError("dataset must be 'breast_cancer' or 'diabetes'")

    stratify_labels = y if stratify else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_labels,
    )

    return X_train, X_test, y_train, y_test, feature_names, target_names


# ============================================================
# EXAMPLE USAGE
# ============================================================

X_train, X_test, y_train, y_test, feature_names, target_names = sample_data()

print("Feature names:", feature_names)
print("Target classes:", target_names)
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


In [ ]:
import lingam
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)

causal_model = lingam.DirectLiNGAM()
causal_model.fit(X_train_scaled)

adjacency_matrix = causal_model.adjacency_matrix_

pbounds = {col: (X_train_scaled[col].min(), X_train_scaled[col].max()) for col in X_train.columns}


In [ ]:
from graphviz import Source
from IPython.display import display
import numpy as np

def plot_causal_graph(adjacency_matrix, feature_names, threshold=0.1, dot_path="lingam_causal_graph.dot"):
    from graphviz import Digraph

    dot = Digraph(comment="Learned Causal Graph (LiNGAM)")
    dot.attr(rankdir="LR")
    dot.attr("node", shape="ellipse", style="filled", fillcolor="white", fontname="Helvetica", fontsize="10")
    dot.attr("edge", fontname="Helvetica", fontsize="9")

    n = len(feature_names)

    # --- find active nodes ---
    active_nodes = set()
    for i in range(n):
        for j in range(n):
            if abs(adjacency_matrix[i, j]) >= threshold:
                active_nodes.add(i)
                active_nodes.add(j)

    if not active_nodes:
        print(f"No edges above threshold {threshold}. Nothing to display.")
        return dot

    # --- add nodes ---
    for i in active_nodes:
        dot.node(f"node{i}", feature_names[i])

    # --- add edges ---
    for i in active_nodes:
        for j in active_nodes:
            weight = adjacency_matrix[i, j]
            if abs(weight) >= threshold:
                color = "green" if weight > 0 else "red"
                dot.edge(f"node{i}", f"node{j}", label=f"{weight:.2f}", color=color)

    # --- save DOT file ---
    dot.save(dot_path)
    print(f"DOT file saved as '{dot_path}'")

    # --- display in Jupyter ---
    src = Source.from_file(dot_path)
    display(src)

    return dot


In [ ]:

import numpy as np
import networkx as nx

# Build directed graph
G = nx.DiGraph()
n = adjacency_matrix.shape[0]

for i in range(n):
    for j in range(n):
        if adjacency_matrix[i, j] != 0:
            G.add_edge(j, i)  # j → i (parent → child)

# Topological sort
causal_order = list(nx.topological_sort(G))
print("Causal order:", causal_order)


In [ ]:
graph = plot_causal_graph(adjacency_matrix, feature_names, threshold=0.2)
#display(graph)

In [ ]:
features_order = X_train_scaled.columns.tolist()

query_instance = X_train_scaled.sample(1).values

explainer.fit(X=X_train_scaled, adjacency_matrix=adjacency_matrix, causal_order=causal_order, pbounds=pbounds,
              y=y_train, features_order=features_order,
              masked_features=["age"]
              )

In [ ]:


original_class = model.predict([query_instance])[0]

cf = explainer.counterfactual(desired_class=1-original_class, query_instance=query_instance,plausibility_weight=0.9, diversity_weight=0.1, sparsity_weight=0.1,init_points=100)


In [ ]:
import pandas as pd

# If your instances are 2D arrays (1 sample)
# query_instance: original scaled instance
# cf: counterfactual scaled instance

# Convert to original units
query_original = scaler.inverse_transform(query_instance)
cf_original = scaler.inverse_transform(cf)

# Convert to pandas for readability
feature_names = X_train.columns.tolist()  # or your saved feature names
query_df = pd.DataFrame(query_original, columns=feature_names)
cf_df = pd.DataFrame(cf_original, columns=feature_names)

# Print nicely
print("Original instance (unscaled):")
print(query_df.values)
print("\nGenerated counterfactual (unscaled):")
print(cf_df.values)


# Plot conunterfactuals

In [ ]:

import numpy as np
import pandas as pd

def make_counterfactual_delta_table(
    query_df: pd.DataFrame,
    cf_df: pd.DataFrame,
    decimals: int = 3,
    tol: float = 1e-12,
    cf_index_prefix: str = "CF #"
):
    """
    Build a styled DataFrame indicating how much each feature should change
    to obtain each counterfactual, with signed formatting like +0.232.

    Parameters
    ----------
    query_df : pd.DataFrame
        The original instance(s), unscaled. If it contains 1 row, it will be broadcast
        to the number of rows in cf_df.
    cf_df : pd.DataFrame
        The generated counterfactual instance(s), unscaled. Must have the same columns as query_df.
        Each row is a distinct counterfactual.
    decimals : int
        Number of decimal places to display for deltas.
    tol : float
        Absolute tolerance under which a delta is considered zero (displayed as blank).
    cf_index_prefix : str
        Prefix for counterfactual row names.

    Returns
    -------
    styled : pd.io.formats.style.Styler
        A styled table with signed deltas and color cues (green for increases, red for decreases).
    delta_df : pd.DataFrame
        The numeric delta DataFrame (cf_df - query_df).
    """

    # --- Basic validation
    if not isinstance(query_df, pd.DataFrame) or not isinstance(cf_df, pd.DataFrame):
        raise TypeError("query_df and cf_df must be pandas DataFrames")

    if list(query_df.columns) != list(cf_df.columns):
        raise ValueError("query_df and cf_df must have identical columns (same order).")

    # Broadcast query row if a single instance is provided
    if len(query_df) == 1 and len(cf_df) > 1:
        query_aligned = pd.DataFrame(
            np.repeat(query_df.values, repeats=len(cf_df), axis=0),
            columns=query_df.columns,
            index=cf_df.index
        )
    else:
        # Otherwise, they must have the same number of rows
        if len(query_df) != len(cf_df):
            raise ValueError(
                f"Row mismatch: query_df has {len(query_df)} rows, cf_df has {len(cf_df)} rows. "
                f"Provide one query row to broadcast or match the counts."
            )
        query_aligned = query_df.copy()
        query_aligned.index = cf_df.index

    # Compute deltas (what to add to the original to reach the CF)
    delta = cf_df - query_aligned

    # Index: label rows as CF #1, CF #2, ...
    delta.index = [f"{cf_index_prefix}{i+1}" for i in range(len(delta))]

    # Format as signed strings (blank if ~0)
    def _fmt_signed(x):
        if pd.isna(x):
            return ""
        if abs(x) < tol:
            return ""
        s = f"{x:+.{decimals}f}"
        # Strip trailing zeros while preserving sign and at least one decimal if necessary
        # Uncomment next line if you prefer trimming (optional):
        # s = s.replace("+", " +").replace("-", "–")  # en-dash for negative if you like
        return s

    formatted = delta.applymap(_fmt_signed)

    # Styling: green for positive, red for negative, gray for zero/blank
    def color_changes(val):
        if val == "":
            return "color: #888888;"  # gray for (near-)zero
        return "color: #2e7d32;" if val.strip().startswith("+") else "color: #c62828;"

    styled = (
        formatted.style
        .applymap(color_changes)
        .set_properties(**{
            "font-family": "Segoe UI, Roboto, Arial, sans-serif",
            "font-size": "12.5px",
            "white-space": "nowrap",
        })
        .set_table_styles([
            {"selector": "th", "props": [("font-weight", "600"), ("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
        ])
        .set_caption("Required feature adjustments to obtain each counterfactual (unscaled)")
    )

    return styled, delta


def counterfactual_instructions(delta_df: pd.DataFrame, tol: float = 1e-12, decimals: int = 3):
    """
    Build natural-language instructions for each counterfactual row:
    Example: 'Increase X by 0.232; decrease Y by 1.50'

    Returns
    -------
    dict: {row_label: instruction_str}
    """
    instructions = {}
    for idx, row in delta_df.iterrows():
        parts = []
        for col, v in row.items():
            if pd.isna(v) or abs(v) < tol:
                continue
            direction = "increase" if v > 0 else "decrease"
            parts.append(f"{direction} {col} by {abs(v):.{decimals}f}")
        instructions[idx] = "; ".join(parts) if parts else "No changes needed"
    return instructions


# -------------------------
# Example usage (adjust to your variables)
# -------------------------

# Your existing prints:
print("Original instance (unscaled):")
print(query_df.values)
print("\nGenerated counterfactual (unscaled):")
print(cf_df.values)

# Pretty delta table
styled, delta_df = make_counterfactual_delta_table(query_df, cf_df, decimals=3)

# If you're in a Jupyter/Notebook environment, display the styled table:
try:
    from IPython.display import display
    display(styled)
except Exception:
    # Fallback for plain consoles: print a text version
    print("\nRequired adjustments (delta = CF - Original):")
    print(delta_df.to_string(float_format=lambda x: f"{x:+.3f}" if abs(x) > 1e-12 else " 0.000"))

# Optional: natural-language instructions per counterfactual
instr = counterfactual_instructions(delta_df, decimals=3)
print("\nInstructions:")
for k, v in instr.items():
    print(f"  {k}: {v}")
